Constitution Generator
Generate a synthetic constitution for a country via a local LLM. The constitution is structured with **sections** and **subsections** to guide the code and conduct of citizens (rights, duties, governance, rule of law, etc.).

Features

- Output as a structured document with numbered sections and subsections (Markdown).
- You can specify the country name, focus areas (e.g. federalism, human rights, judiciary), and length.
- Runs on a single GPU; 4-bit quantization (BitsAndBytes) keeps memory low so it fits on Colab A100 free tier or similar.
- Built with Hugging Face Transformers; responses stream token-by-token in the UI.

In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio

In [ ]:
# Model and generation settings
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 16384  # Constitutions can be long

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer
from huggingface_hub import login
# from google.colab import userdata only required for colab
import os 
from dotenv import load_dotenv 
import torch
import gradio as gr
from threading import Thread

In [ ]:
# hf_token = userdata.get("HF_TOKEN") for colab
load_dotenv() 
hf_token = os.getenv("HF_TOKEN") 
login(hf_token, add_to_git_credential=True)

In [ ]:
def load_model_and_tokenizer(model_id):
    """Load model with 4-bit quantization for lower VRAM (e.g. Colab A100)."""
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config)
    return tokenizer, model

In [ ]:
print("Loading model and tokenizer...")
tokenizer, model = load_model_and_tokenizer(MODEL_ID)
print("Ready.")

In [ ]:
# System prompt
CONSTITUTION_SYSTEM_PROMPT = """You are a constitutional scholar. Your job is to generate a synthetic constitution for a country that guides the code and conduct of citizens.

RULES:
- Only fulfill requests for drafting or generating a constitution (for a named or fictional country). Politely decline unrelated requests (e.g. code, recipes, other topics).
- Structure the constitution with clear SECTIONS (e.g. "Chapter I", "Part II") and SUBSECTIONS (e.g. "Article 1(1)", "Section 5.2"). Use numbered articles and clauses where appropriate.
- Include typical constitutional content: preamble; fundamental rights and duties of citizens; structure of government (executive, legislature, judiciary); rule of law; amendment process; and any other areas the user requests.
- Output in Markdown: use headers for sections (## or ###), bold for article numbers, and plain text for provisions. No informal intro—start with the document title and preamble unless the user asks otherwise.
- If the user does not specify a country name, use a plausible fictional name or "the Republic" as placeholder.
- Keep the tone formal and legal; ensure provisions are coherent and mutually consistent."""

def build_messages(history, user_prompt):
    messages = [{"role": "system", "content": CONSTITUTION_SYSTEM_PROMPT}]
    messages += [{"role": h["role"], "content": h["content"]} for h in (history or [])]
    messages.append({"role": "user", "content": user_prompt})
    return messages

def create_inputs(tokenizer, history, prompt):
    messages = build_messages(history, prompt)
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

In [ ]:
def chat(prompt, history):
    inputs = create_inputs(tokenizer, history, prompt)
    if not isinstance(inputs, dict):
        inputs = {"input_ids": inputs}
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        decode_kwargs={"skip_special_tokens": True},
    )
    thread = Thread(
        target=model.generate,
        kwargs={
            **inputs,
            "max_new_tokens": MAX_NEW_TOKENS,
            "streamer": streamer,
        },
    )
    thread.start()
    full_response = ""
    for chunk in streamer:
        full_response += chunk.replace("<|eot_id|>", "")
        yield full_response

In [ ]:
# Gradio UI: chat to request a constitution (sections and subsections)
with gr.Blocks(title="Constitution Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("Ask for a **constitution** for a country. Specify the country name, focus areas (e.g. human rights, federalism, judiciary), and desired length. The output will have **sections** and **subsections** to guide the code and conduct of citizens.")
    chatbot = gr.Chatbot(height=420, type="messages", label="Chat")
    msg = gr.Textbox(placeholder="e.g. Generate a constitution for the Republic of Atlantica with 8 chapters, emphasis on human rights and separation of powers", label="Request", lines=2)
    submit = gr.Button("Generate", variant="primary")

    def respond(message, history):
        history = list(history or [])
        user_msg = {"role": "user", "content": message}
        for full_reply in chat(message, history):
            yield history + [user_msg, {"role": "assistant", "content": full_reply}]

    submit.click(respond, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)
    msg.submit(respond, inputs=[msg, chatbot], outputs=chatbot).then(lambda: "", None, msg)

demo.queue()
demo.launch()